In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.integrate import solve_ivp

# Example 4.1 — Three-Point Collapse under t-SNE Gradient Flow

**Thesis correspondence:** Section 4.1

This notebook verifies the theoretical analysis of a symmetric 3-point system
{y₁, y₂, y₃} with uniform affinities p_{ij} = 1/6.  We show two equivalent
formulations:

1. **Discrete gradient descent** — iterates the t-SNE update rule directly.
2. **Continuous ODE** — derives and numerically solves ẋ = f(x) where
   y₁(t) = x(t), y₂(t) = 0, y₃(t) = −x(t) by symmetry.

Both paths converge to the same equilibrium: y₁ = −y₃, y₂ = 0,
confirming the theoretical prediction that all three points collapse to a
single location.

## Part 1 — Discrete Gradient Descent

Simulate t-SNE update steps directly on the 3-point system.

# Example 4.1

## Part 2 — Continuous ODE Formulation

By the symmetry y₁ = x, y₂ = 0, y₃ = −x the full gradient system reduces
to the scalar ODE ẋ = f(x) derived below.  We solve it with `scipy.integrate.solve_ivp`
and compare the trajectory to the discrete simulation above.

## Summary

Both the discrete and continuous simulations confirm collapse to equilibrium,
validating the theoretical analysis in Section 4.1 of the thesis.

In [7]:
# Solve the ODE
# This solve_ivp method is a direct analog to ODE45 in MATLAB
sol = solve_ivp(
    fun=lambda t, X: dX_dt(X), # uses our function above
    t_span=[0, 100], # Time interval to solve: from 0 to 50
    y0=[1.0], # initial condition X(0) = 1
    t_eval=np.linspace(0, 50, 500) # time point that are evaluated for the solution
)
X_vals = sol.y[0] # sol.y is the set of X(t) values

# Build full 3-point configuration y(t)
# y1 = X(t), y2 = 0, y3 = -X(t)
history = np.array([[X, 0.0, -X] for X in X_vals])

# Animate in Plotly
def plot_tsne(history):
    frames = []
    y_values = [0, 0, 0]  # All at y=0 for 1D plot
    colors = ['red', 'blue', 'green']

    for i in range(len(history)):
        frames.append(go.Frame(
            data=[go.Scatter(
                x=history[i],
                y=y_values,
                mode="markers",
                marker=dict(size=10, color=colors, opacity=0.8)
            )],
            name=f"Step {i}"
        ))

    fig = go.Figure(
        data=[go.Scatter(
            x=history[0],
            y=y_values,
            mode="markers",
            marker=dict(size=10, color=colors, opacity=0.8)
        )],
        layout=go.Layout(
            title="Example 4.1 – Collapse of Three Points under t-SNE Gradient Flow",
            xaxis_title="Position on Number Line",
            xaxis=dict(title="Position", range=[-1.2, 1.2]),
            yaxis=dict(visible=False),
            updatemenus=[dict(
                type="buttons",
                showactive=False,
                buttons=[
                    dict(label="Play", method="animate",
                         args=[None, dict(frame=dict(duration=30, redraw=True), fromcurrent=True)]),
                    dict(label="Pause", method="animate",
                         args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])
                ]
            )]
        ),
        frames=frames
    )
    fig.show()

plot_tsne(history)
